# DAY 158 - Advanced Text Processing.
@A.IPYNB

### The Core Problem
Computers cannot understand text. They only understand numbers.
"Text Processing" isn't just cleaning; it's **Transformation**.

We will explore three generations of text representation:

1.  **Bag of Words (TF-IDF):** "Does this word appear?"
    * *Pros:* Fast, interpretable.
    * *Cons:* Ignores context ("bank" of a river = "bank" for money).
2.  **Word Embeddings (Word2Vec/GloVe):** "What is this word near?"
    * *Pros:* Captures semantic similarity (King - Man + Woman = Queen).
    * *Cons:* Static (One vector per word, regardless of context).
3.  **Contextual Embeddings (Transformers):** "What does this sentence mean?"
    * *Pros:* State-of-the-art semantic understanding.
    * *Cons:* Computational cost.

In [1]:
# @title 1. Libraries
!pip install -q scikit-learn gensim sentence-transformers nltk

import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
import re

nltk.download('stopwords')
nltk.download('punkt')

print("Libraries installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 57.0 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Libraries installed.


### 1. TF-IDF (Term Frequency - Inverse Document Frequency)
This is the baseline for any NLP project. It highlights words that are **frequent in a document** but **rare across the corpus** (like "mitochondria" in a biology paper), while penalizing common words (like "the").

$$w_{i,j} = tf_{i,j} \times \log(\frac{N}{df_i})$$

In [2]:
# @title 2. TF-IDF Implementation
from sklearn.feature_extraction.text import TfidfVectorizer

# Sample Corpus
corpus = [
    "The quick brown fox jumps over the lazy dog.",
    "The dog is lazy and sleeps all day.",
    "Quick foxes are fast animals.",
    "Machine learning involves vectors and matrices."
]

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(corpus)
feature_names = vectorizer.get_feature_names_out()
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

print("TF-IDF Matrix (Rows=Documents, Cols=Words):")
# We'll Show only the first few columns for readability
print(df_tfidf.iloc[:, :8])

TF-IDF Matrix (Rows=Documents, Cols=Words):
    animals     brown       day       dog      fast       fox     foxes  \
0  0.000000  0.453386  0.000000  0.357455  0.000000  0.453386  0.000000   
1  0.000000  0.000000  0.555283  0.437791  0.000000  0.000000  0.000000   
2  0.525473  0.000000  0.000000  0.000000  0.525473  0.000000  0.525473   
3  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   

   involves  
0  0.000000  
1  0.000000  
2  0.000000  
3  0.447214  


### 2. Word2Vec (Dense Embeddings)
Instead of a sparse matrix of 0s and 1s, we map every word to a dense vector (e.g., 100 dimensions).
Words that appear in similar contexts will be close together in this vector space.

We will use a pre-trained **GloVe** model (Global Vectors for Word Representation) to perform "Vector Arithmetic".

In [3]:
# @title 3. Loading GloVe Embeddings
import gensim.downloader as api

# Here we'll Download a small pre-trained model (Twitter 25-dimension vectors)
# In production, we can use 'word2vec-google-news-300' (1.6GB)
print("Downloading GloVe model (this may take a minute)...")
model_glove = api.load("glove-twitter-25")

print("Model Loaded.")

# "King" - "Man" + "Woman" should equal "Queen"
try:
    result = model_glove.most_similar(positive=['woman', 'king'], negative=['man'], topn=1)
    print(f"\nKing - Man + Woman = {result[0][0]} (Similarity: {result[0][1]:.2f})")
except KeyError:
    print("Words not found in this small vocabulary.")

word1 = "computer"
word2 = "laptop"
sim = model_glove.similarity(word1, word2)
print(f"Similarity between '{word1}' and '{word2}': {sim:.2f}")

[==================================================] 100.0% 104.8/104.8MB downloaded
Model Loaded.

King - Man + Woman = meets (Similarity: 0.88)
Similarity between 'computer' and 'laptop': 0.84


### Analyzing the Results (Model Size Matters)

**1. The Success:**
* **Similarity:** `0.84` between "computer" and "laptop".
* **Verdict:** Even a tiny 25-dimensional model trained on Tweets understands basic synonyms.

**2. The Failure:**
* **Analogy:** King - Man + Woman = `meets` (Similarity: 0.88).
* **Expected:** "Queen".
* **The Lesson:** This is why model size matters. We loaded `glove-twitter-25` (only 25 dimensions) for speed. It is too small to capture complex gender relationships.
    * *To fix this in production:* You would load `word2vec-google-news-300` (300 dimensions, 1.6GB), which perfectly solves the "Queen" analogy.

### 3. Sentence Transformers (SBERT)
This is the modern standard. Instead of averaging word vectors (which loses grammar), SBERT uses a Transformer (BERT) to encode the **entire sentence** into a single 384-dimensional vector.

This allows for **Semantic Search**. We can search for queries that *mean* the same thing, even if they don't share keywords.

In [4]:
# @title 4. Building a Semantic Search Engine
from sentence_transformers import SentenceTransformer, util

model_sbert = SentenceTransformer('all-MiniLM-L6-v2')

# A database of "documents"
docs = [
    "A man is eating food.",
    "A man is eating a piece of bread.",
    "The girl is carrying a baby.",
    "A man is riding a horse.",
    "A woman is playing violin.",
    "Two men pushed carts through the woods.",
    "A man is riding a white horse on an enclosed ground.",
    "A monkey is playing drums.",
    "Someone in a gorilla suit is playing a set of drums."
]

doc_embeddings = model_sbert.encode(docs, convert_to_tensor=True)

query = "A primate playing music"
query_embedding = model_sbert.encode(query, convert_to_tensor=True)

# Here I'm Computing Cosine Similarity
# We compare the Query Vector against ALL Document Vectors instantly
hits = util.semantic_search(query_embedding, doc_embeddings, top_k=3)

print(f"Query: {query}\n")
print("Top 3 Semantic Matches:")
for hit in hits[0]:
    print(f"Score: {hit['score']:.4f} | Text: {docs[hit['corpus_id']]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Query: A primate playing music

Top 3 Semantic Matches:
Score: 0.6054 | Text: A monkey is playing drums.
Score: 0.5553 | Text: Someone in a gorilla suit is playing a set of drums.
Score: 0.3631 | Text: A woman is playing violin.


### The Strategic Win: Semantic Search

We have just witnessed the power of **Contextual Embeddings**.

**The Evidence from the Output:**
* **The Query:** "A primate playing music"
* **The Result:** "A monkey is playing drums." (Score: 0.6054)

**Why this is a breakthrough:**
1.  **No Keyword Overlap:** The word "Primate" is not in the result. The word "Music" is not in the result.
2.  **True Understanding:** The model understood that a *Monkey* is a type of *Primate*, and *Drums* are a type of *Music*.

**The Business Value:**
Traditional search would have failed here. By using SBERT, you can build search engines that find what users *mean*, not just what they *type*.